<a href="https://colab.research.google.com/github/lady-trie/PraktikumStatistika/blob/main/Uji_Validitas_Reliabilitas_Statistika.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [2]:
# 1. Upload file
up = files.upload()
file_name = list(up.keys())[0]
df = pd.read_excel(file_name)

Saving Blank Quiz (Responses) (1).xlsx to Blank Quiz (Responses) (1).xlsx


In [3]:
# 2. Separasi data demografi & likert
JUMLAH_DEMOGRAFI = 6  # sesuaikan jumlah kolom demografi di excel kalian
butir = df.iloc[:, JUMLAH_DEMOGRAFI:].apply(lambda x: x.astype(str).str.strip() if x.dtype == "object" else x)

In [4]:
# 3. Konversi ke angka
skala = {
    "Sangat Tidak Setuju (STS)": 1, "Tidak Setuju (TS)": 2, "Netral (N)": 3,
    "Setuju (S)": 4, "Sangat Setuju (SS)": 5,
    "Sangat Tidak Setuju": 1, "Tidak Setuju": 2, "Netral": 3, "Setuju": 4, "Sangat Setuju": 5
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

/tmp/ipykernel_1624/3476708527.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


In [5]:
# 4. Pengelompokan dimensi
dimensi = {}
for col in num.columns:
    m = re.match(r"^(.*?)\[", col)
    nama = m.group(1).strip() if m else "Umum"
    dimensi.setdefault(nama, []).append(col)

In [6]:
# 5. Uji Validitas (Pearson & Corrected Item-Total)
n = len(num)
r_tabel = stats.t.ppf(0.975, n - 2) / np.sqrt((n - 2) + stats.t.ppf(0.975, n - 2)**2)

val_list = []
for nama, cols in dimensi.items():
    sub = num[cols]
    tot = sub.sum(axis=1)
    for i, col in enumerate(cols, 1):
        r, p = stats.pearsonr(sub[col], tot)
        r_corr, _ = stats.pearsonr(sub[col], tot - sub[col])
        val_list.append({
            "Dimensi": nama, "Butir": f"B{i}",
            "r hitung": round(r, 3), "Corrected r": round(r_corr, 3), "p-value": round(p, 4),
            "Status": "Valid" if r > r_tabel else "Tidak Valid"
        })

df_validitas = pd.DataFrame(val_list)

/tmp/ipykernel_1624/3948190979.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(sub[col], tot)


In [7]:
# 6. Uji Reliabilitas (Cronbach Alpha)
def get_alpha(df_in):
    k = df_in.shape[1]
    if k <= 1: return np.nan
    return (k / (k - 1)) * (1 - (df_in.var(ddof=1).sum() / df_in.sum(axis=1).var(ddof=1)))

rel_list, del_list = [], []
for nama, cols in dimensi.items():
    sub = num[cols]
    a = get_alpha(sub)
    kat = "Sangat Baik" if a>=0.9 else "Baik" if a>=0.8 else "Reliabel" if a>=0.7 else "Tidak Reliabel"
    rel_list.append({"Dimensi": nama, "Butir": len(cols), "Cronbach Alpha": round(a, 3), "Kategori": kat})

    if len(cols) > 2:
        for i, col in enumerate(cols, 1):
            del_list.append({
                "Dimensi": nama, "Butir": f"B{i}",
                "Alpha if Deleted": round(get_alpha(sub.drop(columns=[col])), 3)
            })

df_reliabilitas = pd.DataFrame(rel_list)
df_del = pd.DataFrame(del_list)

In [10]:
# Output Tampilan
print(f"r tabel (n={n}, df={n-2}) = {r_tabel:.3f}\n")
display(df_reliabilitas)
display(df_validitas.merge(df_del, on=["Dimensi", "Butir"], how="left"))

r tabel (n=37, df=35) = 0.325



,Dimensi,Butir,Cronbach Alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.858,Baik
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.904,Sangat Baik
2,Umum,1,NaN,Tidak Reliabel
3,Kebiasaan penggunaan teknologi digital,6,0.777,Reliabel
4,Prokrastinasi akademik.,4,0.846,Baik
5,Rsspon emosional,5,0.875,Baik


,Dimensi,Butir,r hitung,Corrected r,p-value,Status,Alpha if Deleted
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.793,0.701,0.0000,Valid,0.827
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.732,0.624,0.0000,Valid,0.838
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.790,0.703,0.0000,Valid,0.827
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.433,0.295,0.0075,Valid,0.875
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.764,0.650,0.0000,Valid,0.834
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.838,0.755,0.0000,Valid,0.818
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.746,0.627,0.0000,Valid,0.838
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.862,0.801,0.0000,Valid,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.790,0.712,0.0000,Valid,0.890
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.779,0.682,0.0000,Valid,0.894


In [9]:
# Export Excel
out_name = "Hasil_Analisis.xlsx"
with pd.ExcelWriter(out_name) as w:
    df_validitas.merge(df_del, on=["Dimensi", "Butir"], how="left").to_excel(w, sheet_name="Validitas", index=False)
    df_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download(out_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>